# Retrieval Augmented Generation

### setup vector store

In [1]:
!pip install -qU langchain-chroma

In [2]:
from langchain_chroma import Chroma

In [3]:
from langchain_openai import OpenAIEmbeddings

In [5]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")

In [6]:
db = Chroma(
    persist_directory = "./chroma_db",
    embedding_function=embedding_model
)

### Retrieve

In [7]:
retriever = db.as_retriever(
    search_type="mmr", search_kwargs={"k": 1, "fetch_k": 5, "lambda_mult":0.8}
)

In [9]:
question = "화석은 진화론을 뒷받침하고 있나요?"

In [10]:
context = retriever.invoke(question)

In [11]:
context[0].metadata

{'source': 'http://creation.kr/Circulation/?idx=1294861&bmode=view'}

### Augment

In [12]:
from langchain.prompts import ChatPromptTemplate
from langchain.prompts import HumanMessagePromptTemplate
from langchain_core.messages import SystemMessage

In [24]:
chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(
            content= "당신은 지적설계를 지지하는 과학자입니다."
        ),
        HumanMessagePromptTemplate.from_template(
            """
            {question}
            아래의 문맥에 기반하여 답하여 주세요.
            {context}

            문맥에서 언급되었다는 투의 말을 생략하고 응답을 제공해주세요.
            """
        )
    ]
)

In [25]:
message = chat_template.format_messages(
    question=question,
    context=context[0].page_content
)

### Generate

In [26]:
from langchain_openai import ChatOpenAI

In [27]:
model = ChatOpenAI(
    model_name = "gpt-4o-mini",
    temperature = 0
)

In [28]:
for chunk in model.stream(message):
    print(chunk.content, end="", flush=True)

화석 기록은 진화론을 뒷받침하지 않는다는 주장이 있습니다. 진화론자들은 화석들이 지질 기록에서 일반적으로 순서를 가지고 나타나는 이유를 설명해야 한다고 요구하지만, 창조론자들은 전이형태의 생물들이 존재하지 않는 이유와 특정 생물들이 화석으로 발견되지 않는 이유에 대한 설명을 요구해야 한다고 주장합니다. 또한, 생물종의 안정성이 흔하다는 점을 강조하며, 변함이 없었던 생물체의 화석이 반복적으로 발견되어야 한다고 지적합니다. 이러한 관점은 화석 기록이 진화론의 주장과 일치하지 않는다는 주장을 뒷받침합니다.